In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
import ast
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA

plt.rcParams['figure.dpi'] = 120
print('Libraries loaded ✅')

## 1. Load the Dataset

In [2]:
df = pd.read_csv('tmdb_5000_movies.csv')

print(f'Shape: {df.shape}')
print(f'\nColumns: {df.columns.tolist()}')
print(f'\nMissing values:\n{df.isnull().sum()}')
df.head()

## 2. Feature Engineering

The dataset contains several JSON columns (`genres`, `keywords`, `production_companies`, `production_countries`, `spoken_languages`) that we parse to extract numeric features.

We also handle zero values in `budget` and `revenue` (treated as missing) and engineer the following features:

- `budget` — production budget (zeros excluded)
- `revenue` — box office revenue (zeros excluded)
- `runtime` — film duration in minutes
- `popularity` — TMDB popularity score
- `vote_average` — average user rating
- `vote_count` — number of votes
- `num_genres` — number of genres
- `num_keywords` — number of keywords
- `num_production_companies` — number of production companies
- `num_countries` — number of production countries
- `num_languages` — number of spoken languages
- `is_english` — 1 if original language is English

In [3]:
def parse_names(x):
    try:
        return [d['name'] for d in ast.literal_eval(x)]
    except:
        return []

def parse_count(x):
    try:
        return len(ast.literal_eval(x))
    except:
        return 0

# Parse JSON columns
df['genres_list']    = df['genres'].apply(parse_names)
df['keywords_list']  = df['keywords'].apply(parse_names)

df['num_genres']               = df['genres'].apply(parse_count)
df['num_keywords']             = df['keywords'].apply(parse_count)
df['num_production_companies'] = df['production_companies'].apply(parse_count)
df['num_countries']            = df['production_countries'].apply(parse_count)
df['num_languages']            = df['spoken_languages'].apply(parse_count)
df['is_english']               = (df['original_language'] == 'en').astype(int)

# Replace 0 with NaN for budget and revenue
df['budget']  = df['budget'].replace(0, np.nan)
df['revenue'] = df['revenue'].replace(0, np.nan)

feature_cols = ['budget', 'revenue', 'runtime', 'popularity',
                'vote_average', 'vote_count',
                'num_genres', 'num_keywords',
                'num_production_companies', 'num_countries',
                'num_languages', 'is_english']

print('Features engineered:')
print(df[feature_cols].describe().round(2))

## 3. Preprocessing

We drop rows with missing values in any feature column, then apply `StandardScaler` to normalise all features before clustering.

In [4]:
df_clean = df[feature_cols + ['title']].dropna().reset_index(drop=True)

print(f'Rows before dropna : {len(df)}')
print(f'Rows after  dropna : {len(df_clean)}')
print(f'Rows dropped       : {len(df) - len(df_clean)}')

X = df_clean[feature_cols]
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'\nScaled matrix shape: {X_scaled.shape}')
print(f'Mean (~0): {X_scaled.mean(axis=0).round(3)}')
print(f'Std  (~1): {X_scaled.std(axis=0).round(3)}')

## 4. K-Means Clustering

### Elbow Method / SSE & Silhouette Score

We run K-Means for k = 2 to 10 and plot **SSE (inertia)** and **Silhouette score** to find the optimal number of clusters.

In [5]:
sse                  = []
silhouette_scores_km = []
K_range              = range(2, 11)

for k in K_range:
    km     = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    sse.append(km.inertia_)
    silhouette_scores_km.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(K_range, sse, 'o-', color='steelblue', linewidth=2, markersize=7)
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('SSE (Inertia)')
axes[0].set_title('Elbow Method — SSE vs k', fontweight='bold')
axes[0].grid(alpha=0.3)

axes[1].plot(K_range, silhouette_scores_km, 's-', color='darkorange', linewidth=2, markersize=7)
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score vs k', fontweight='bold')
axes[1].grid(alpha=0.3)

plt.suptitle('K-Means — Optimal k Selection', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f'{"k":<5} {"SSE":>12}  {"Silhouette":>12}')
print('-' * 35)
for k, s, sil in zip(K_range, sse, silhouette_scores_km):
    print(f'k={k}   {s:>10.1f}   {sil:.4f}')

In [6]:
best_k = K_range[silhouette_scores_km.index(max(silhouette_scores_km))]
print(f'Best k by Silhouette: {best_k}  (score={max(silhouette_scores_km):.4f})')

km_final  = KMeans(n_clusters=best_k, random_state=42, n_init=10)
km_labels = km_final.fit_predict(X_scaled)
df_clean['kmeans_cluster'] = km_labels

print(f'\nCluster sizes:')
print(df_clean['kmeans_cluster'].value_counts().sort_index())

### K-Means Cluster Visualisation (PCA 2D)

In [7]:
pca   = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

colors_km = ['#4C9BE8', '#F4845F', '#57B894', '#9C27B0', '#FF9800',
             '#E53935', '#00BCD4', '#8BC34A', '#FF5722']

fig, ax = plt.subplots(figsize=(9, 6))
for cluster in sorted(df_clean['kmeans_cluster'].unique()):
    mask = df_clean['kmeans_cluster'] == cluster
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=colors_km[cluster], alpha=0.4, s=12, label=f'Cluster {cluster}')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.set_title(f'K-Means (k={best_k}) — PCA 2D Projection', fontweight='bold')
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print(f'Variance explained by 2 PCs: {pca.explained_variance_ratio_.sum()*100:.1f}%')

### K-Means Cluster Profiles

In [8]:
profile = df_clean.groupby('kmeans_cluster')[feature_cols].mean().round(2)
print('Mean feature values per cluster:')
print(profile.to_string())

### Sample Movies per Cluster

In [9]:
for cluster in sorted(df_clean['kmeans_cluster'].unique()):
    sample = df_clean[df_clean['kmeans_cluster'] == cluster]['title'].head(5).tolist()
    print(f'Cluster {cluster}: {sample}')

## 5. DBSCAN Clustering

DBSCAN does not require specifying the number of clusters upfront. It identifies dense regions and marks sparse points as **noise (label = -1)**.

We run a grid search over `eps` and `min_samples` and select the configuration with the best Silhouette score on non-noise points.

In [10]:
eps_values         = [0.5, 0.8, 1.0, 1.2, 1.5, 2.0]
min_samples_values = [5, 10, 15, 20]

print(f'{"eps":<8} {"min_samples":<14} {"clusters":<12} {"noise%":<10} {"silhouette"}')
print('-' * 62)

best_sil_db    = -1
best_params    = {}
best_db_labels = None

for eps in eps_values:
    for ms in min_samples_values:
        db     = DBSCAN(eps=eps, min_samples=ms)
        labels = db.fit_predict(X_scaled)
        n_cl   = len(set(labels)) - (1 if -1 in labels else 0)
        noise  = (labels == -1).sum() / len(labels) * 100

        if n_cl >= 2:
            mask = labels != -1
            sil  = silhouette_score(X_scaled[mask], labels[mask])
            print(f'{eps:<8} {ms:<14} {n_cl:<12} {noise:<9.1f}% {sil:.4f}')
            if sil > best_sil_db:
                best_sil_db    = sil
                best_params    = {'eps': eps, 'min_samples': ms}
                best_db_labels = labels.copy()
        else:
            print(f'{eps:<8} {ms:<14} {n_cl:<12} {noise:<9.1f}% —')

print(f'\nBest: eps={best_params["eps"]}, min_samples={best_params["min_samples"]}  →  Silhouette={best_sil_db:.4f}')

### DBSCAN Cluster Visualisation (PCA 2D)

In [11]:
db_labels = best_db_labels
df_clean['dbscan_cluster'] = db_labels

n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
noise_count   = (db_labels == -1).sum()

print(f'DBSCAN — eps={best_params["eps"]}, min_samples={best_params["min_samples"]}')
print(f'Clusters found : {n_clusters_db}')
print(f'Noise points   : {noise_count} ({noise_count/len(db_labels)*100:.1f}%)')
print(f'\nCluster sizes:')
print(df_clean['dbscan_cluster'].value_counts().sort_index())

palette   = ['#4C9BE8', '#F4845F', '#57B894', '#FF9800', '#9C27B0', '#E53935']
color_map = {-1: '#cccccc'}
for i, cl in enumerate([l for l in sorted(set(db_labels)) if l != -1]):
    color_map[cl] = palette[i % len(palette)]

fig, ax = plt.subplots(figsize=(9, 6))
for label in sorted(set(db_labels)):
    mask = db_labels == label
    name = 'Noise' if label == -1 else f'Cluster {label}'
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=color_map[label],
               alpha=0.12 if label == -1 else 0.45,
               s=12, label=name)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.set_title(f'DBSCAN — PCA 2D Projection\neps={best_params["eps"]}, min_samples={best_params["min_samples"]}',
             fontweight='bold')
ax.legend(markerscale=2)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 6. Cluster Evaluation — Silhouette Score

The Silhouette score measures how well each point fits its assigned cluster compared to other clusters. It ranges from **-1 to +1**:

- Close to **+1** → well-separated, compact clusters  
- Close to **0** → overlapping clusters  
- **Negative** → point may be in the wrong cluster

In [12]:
sil_km = silhouette_score(X_scaled, km_labels)

mask_db = db_labels != -1
sil_db  = silhouette_score(X_scaled[mask_db], db_labels[mask_db]) if mask_db.sum() > 1 else None

print(f'{"="*50}')
print(f'  CLUSTER EVALUATION — SILHOUETTE SCORE')
print(f'{"="*50}')
print(f'  K-Means (k={best_k})                    : {sil_km:.4f}')
if sil_db:
    print(f'  DBSCAN  (eps={best_params["eps"]}, ms={best_params["min_samples"]}) : {sil_db:.4f}')
print(f'{"="*50}')
print()
print('Guide:  > 0.70 Strong  |  0.50-0.70 Reasonable  |  0.25-0.50 Weak  |  < 0.25 No structure')

### Silhouette Plot per Sample — K-Means

In [13]:
sil_vals = silhouette_samples(X_scaled, km_labels)

fig, ax = plt.subplots(figsize=(10, 5))
y_lower    = 10
colors_sil = ['#4C9BE8', '#F4845F', '#57B894', '#9C27B0', '#FF9800',
               '#E53935', '#00BCD4', '#8BC34A', '#FF5722']

for cluster in range(best_k):
    vals    = np.sort(sil_vals[km_labels == cluster])
    y_upper = y_lower + len(vals)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, vals,
                     facecolor=colors_sil[cluster], alpha=0.7)
    ax.text(-0.05, y_lower + len(vals) / 2, f'C{cluster}', fontsize=10)
    y_lower = y_upper + 10

ax.axvline(sil_km, color='red', linestyle='--', label=f'Avg silhouette = {sil_km:.3f}')
ax.set_xlabel('Silhouette coefficient')
ax.set_ylabel('Cluster')
ax.set_title(f'Silhouette Plot — K-Means (k={best_k})', fontweight='bold')
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 7. Summary / Comparison

In [14]:
noise_pct_db = (db_labels == -1).sum() / len(db_labels) * 100

summary_df = pd.DataFrame({
    'Algorithm'      : ['K-Means', 'DBSCAN'],
    'Parameters'     : [f'k={best_k}',
                        f'eps={best_params["eps"]}, min_samples={best_params["min_samples"]}'],
    'Clusters Found' : [best_k, n_clusters_db],
    'Noise Points'   : ['N/A', f'{noise_count} ({noise_pct_db:.1f}%)'],
    'Silhouette'     : [f'{sil_km:.4f}', f'{sil_db:.4f}' if sil_db else 'N/A'],
    'Notes'          : [
        'Requires k upfront; assumes spherical clusters',
        'Finds arbitrary shapes; handles noise automatically'
    ]
})

print('=' * 80)
print('  MODEL COMPARISON SUMMARY')
print('=' * 80)
print(summary_df.to_string(index=False))
print('=' * 80)

In [15]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# K-Means
for cluster in sorted(df_clean['kmeans_cluster'].unique()):
    mask = df_clean['kmeans_cluster'] == cluster
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=colors_km[cluster], alpha=0.4, s=10, label=f'Cluster {cluster}')
axes[0].set_title(f'K-Means (k={best_k}) — Silhouette={sil_km:.3f}', fontweight='bold')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].legend(markerscale=2, fontsize=9)
axes[0].grid(alpha=0.2)

# DBSCAN
for label in sorted(set(db_labels)):
    mask = db_labels == label
    name = 'Noise' if label == -1 else f'Cluster {label}'
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=color_map[label],
                    alpha=0.12 if label == -1 else 0.45,
                    s=10, label=name)
sil_str = f'{sil_db:.3f}' if sil_db else 'N/A'
axes[1].set_title(f'DBSCAN — Silhouette={sil_str}', fontweight='bold')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].legend(markerscale=2, fontsize=9)
axes[1].grid(alpha=0.2)

plt.suptitle('Clustering Comparison — K-Means vs DBSCAN (PCA 2D)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()